In [ ]:
from keras.datasets import imdb

(x_train, y_train), (x_test, y_test) = imdb.load_data()

print(len(x_train))
print(len(x_test))

In [ ]:
print(len(x_train[0]))
print(x_train[0])
print(y_train[0])

In [ ]:
print(len(x_train[1]))
print(x_train[1])
print(y_train[1])

In [ ]:
word_index = imdb.get_word_index()
word_index

In [ ]:
sorted_word_index = dict(sorted(word_index.items(), key=lambda x: x[1]))
sorted_word_index

In [ ]:
reverse_word_index = {value: key for key, value in sorted_word_index.items()}
reverse_word_index

In [ ]:
reverse_word_index.get(11, '')

In [ ]:
positive_indices = []
negative_indices = []

for i in range(len(y_train)):
    if y_train[i]==1:
        positive_indices.append(i)
    else:
        negative_indices.append(i)

In [ ]:
print("Positive Examples:")

for i in range(5):
    
    numbers=x_train[positive_indices[i]]
    
    words=[reverse_word_index.get(idx-3, '') for idx in numbers]
    
    sentence=' '.join(words)
    
    print('*' + sentence)

In [ ]:
print("\nNegative Examples:")

for i in range(5):
    
    numbers = x_train[negative_indices[i]]
    
    words=[reverse_word_index.get(idx-3, '') for idx in numbers]
    
    sentence = ' '.join(words)
    
    print('*' + sentence)

In [ ]:
max_words = max(word_index.values())

print("Maximum number of words:", max_words)

In [ ]:
import numpy as np

positive_count = np.sum(y_train == 1)
negative_count = np.sum(y_train == 0)

print("Number of positive items:", positive_count)
print("Number of negative items:", negative_count)

In [ ]:
max_words = 1000

(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=max_words)

In [ ]:
min_length = float('inf')
max_length = 0

for sequence in x_train:
    sequence_length = len(sequence)
    min_length = min(min_length, sequence_length)
    max_length = max(max_length, sequence_length)

print("Minimum sequence length:", min_length)
print("Maximum sequence length:", max_length)

In [ ]:
print(len(x_train[5]))
print(x_train[5])

In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

max_sequence_length = 500

x_train = pad_sequences(x_train, maxlen=max_sequence_length)
x_test = pad_sequences(x_test, maxlen=max_sequence_length)

In [ ]:
print(len(x_train[5]))
print(x_train[5])

In [ ]:
from keras.models import Sequential
from keras.layers import LSTM, Dense, Embedding, Input

model = Sequential()

model.add(Input(shape=(max_sequence_length,)))

output_dim = 128

model.add(Embedding(input_dim=max_words, input_length=max_sequence_length, output_dim=output_dim))

model.add(LSTM(128))
model.add(Dense(1, activation='sigmoid'))

In [ ]:
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [ ]:
epochs = 20

history = model.fit(x_train, y_train, batch_size=32, epochs=epochs, validation_data=(x_test, y_test))

In [ ]:
train_accuracy = history.history['accuracy']
val_accuracy = history.history['val_accuracy']

print("Train Accuracy:", train_accuracy[-1])
print("Validation Accuracy:", val_accuracy[-1])

In [ ]:
model.save('imdb_trained_model.keras')

In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

def preprocess_input_sentence(sentence):
    words=[]
    ar = sentence.split()
    for x in ar:
        words.append(x.lower())
    sequence = [word_index.get(word, 0) + 3 if word in word_index else '' for word in words]
    sequence = [x if x<=max_words else 0 for x in sequence]
    padded_sequences = pad_sequences([sequence], maxlen=max_sequence_length)
    return padded_sequences

In [ ]:
from keras.models import load_model

def classify_sentence(input_sentence):
    
    preprocessed_sentence = preprocess_input_sentence(input_sentence)
    
    model = load_model('imdb_trained_model.keras')
    
    predictions = model.predict(preprocessed_sentence)
    
    sentiment = "Positive" if predictions[0] > 0.5 else "Negative"
    return sentiment

In [ ]:
new_sentences = ["The food was fantastic", 
                 "The movie was terrible",
                 "I love this movie it is great one"]

predictions = [classify_sentence(sentence) for sentence in new_sentences]


for sentence, prediction in zip(new_sentences, predictions):
    print(f"Sentence: {sentence} - Sentiment: {prediction}  ")